# Generative AI 009 — Chains

Chains are composition, and a stub chat model satisfies the same interface as a
real one — so **every chain here is built and run with no API key**.

| Part | What we check |
|---|---|
| A | `prompt \| model \| parser` is a `RunnableSequence` with **3 steps** |
| B | sequential: two model calls, **6 steps**, and why the middle parser matters |
| C | parallel: **1.99×** measured, and why that is the ceiling |
| D | conditional: exactly one branch, and how it breaks silently |
| E | a 3-step chain's graph has **5 nodes** |

Needs `langchain-core` and `pydantic`.

In [ ]:
import warnings, time
warnings.filterwarnings("ignore")

from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda

## Part A — The simplest chain

In [ ]:
model = FakeListChatModel(responses=["1. Cricket is old. 2. ... 5. ..."])
chain = (PromptTemplate.from_template("Generate five interesting facts about {topic}")
         | model
         | StrOutputParser())

print(chain.invoke({"topic": "cricket"}))
print()
print("type :", type(chain).__name__)
print("steps:", [type(s).__name__ for s in chain.steps])

assert type(chain).__name__ == "RunnableSequence"
assert len(chain.steps) == 3

The chain is an **object**, not syntax. You can print its steps, pass it
around, and — as lesson 010 shows — put it inside another chain.

## Part B — Sequential

In [ ]:
report = PromptTemplate.from_template("Generate a detailed report on {topic}")
summarise = PromptTemplate.from_template(
    "Generate a five-point summary from the following text.\n{text}")

m = FakeListChatModel(responses=[
    "Unemployment in India has many causes ...",
    "1. Jobs. 2. Skills. 3. Growth. 4. Policy. 5. Outlook.",
])
parser = StrOutputParser()

seq = report | m | parser | summarise | m | parser
print("steps:", len(seq.steps))
print(seq.invoke({"topic": "unemployment in India"}))

assert len(seq.steps) == 6

In [ ]:
# Take the MIDDLE parser out. You might expect a type error. Watch.
m2 = FakeListChatModel(responses=["Unemployment in India has many causes ...",
                                  "1. Jobs. 2. Skills."])
no_parser = report | m2 | summarise | m2 | parser
print("result:", repr(no_parser.invoke({"topic": "unemployment in India"})))
print("-> it ran. No error at all.")

In [ ]:
# So what did the second model actually receive?
m3 = FakeListChatModel(responses=["Unemployment in India has many causes ..."])
message = (report | m3).invoke({"topic": "unemployment in India"})
filled = summarise.invoke(message).text
print(repr(filled))
print()

content = message.content
clean = summarise.invoke({"text": content}).text
print(f"real content : {len(content):>4} characters")
print(f"pure noise   : {len(filled) - len(clean):>4} characters")

assert len(filled) > len(clean)          # the repr is strictly longer
assert "additional_kwargs" in filled     # Python internals, in the prompt

The template did not reject the `AIMessage` — it called `str()` on it. The
model was handed the object's **repr**: the content, then `additional_kwargs`,
`response_metadata`, a run id, `tool_calls` and `invalid_tool_calls`, all
pasted in as though they were the report.

**41 characters of real content became 181 characters of prompt.**

It does not raise. The chain returns a plausible answer. You pay for the noise
on every call. The parser is not there to satisfy a type checker — it is there
to stop the model reading Python internals.

## Part C — Parallel, with the timing measured

Replace the model with a stub that **sleeps**, standing in for network latency
— which is what a parallel chain actually saves.

In [ ]:
class SlowModel(FakeListChatModel):
    sleep_for: float = 0.4
    def _call(self, messages, stop=None, run_manager=None, **kwargs):
        time.sleep(self.sleep_for)
        return super()._call(messages, stop=stop, run_manager=run_manager, **kwargs)

notes_chain = (PromptTemplate.from_template("Notes from:\n{text}")
               | SlowModel(responses=["NOTES"]) | StrOutputParser())
quiz_chain  = (PromptTemplate.from_template("Q&As from:\n{text}")
               | SlowModel(responses=["QUIZ"])  | StrOutputParser())

parallel = RunnableParallel({"notes": notes_chain, "quiz": quiz_chain})

t = time.perf_counter(); out = parallel.invoke({"text": "<big text>"}); par = time.perf_counter() - t
t = time.perf_counter()
notes_chain.invoke({"text": "<big text>"}); quiz_chain.invoke({"text": "<big text>"})
seq_t = time.perf_counter() - t

print("parallel result:", out)
print(f"parallel {par:.2f}s   one-then-other {seq_t:.2f}s   speedup {seq_t/par:.2f}x")

assert 1.5 < seq_t / par <= 2.05      # ceiling is the branch count

Two 0.4 s calls: **0.80 s** in sequence, **0.40 s** together. The ceiling is the
**number of branches** — two branches, at best 2×, never quite reached because
starting them costs something.

**This saves latency, not money.** Both calls still happen and you pay for both.

In [ ]:
# RunnableParallel returns a dict whose KEYS are the branch names - and those
# are exactly the placeholders the merge template asks for.
merge = PromptTemplate.from_template("Merge.\nNotes: {notes}\nQuiz: {quiz}")
full = parallel | merge | SlowModel(responses=["MERGED"]) | StrOutputParser()

t = time.perf_counter(); print(full.invoke({"text": "<big text>"})); full_t = time.perf_counter() - t
print(f"{full_t:.2f}s  = one parallel leg + one merge call")
print("The merge needs BOTH branches, so it waits for the slower one.")

## Part D — Conditional

Only **one** branch runs. The interesting part is that the branch depends on a
value the *model* produced — and models are unreliable about exact strings.

In [ ]:
class Feedback(BaseModel):
    """The sentiment of a piece of customer feedback."""
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment")

parser2 = PydanticOutputParser(pydantic_object=Feedback)
classify = PromptTemplate(
    template="Classify this feedback.\n{feedback}\n{format_instructions}",
    input_variables=["feedback"],
    partial_variables={"format_instructions": parser2.get_format_instructions()},
)

branch = RunnableBranch(
    (lambda x: x.sentiment == "positive", RunnableLambda(lambda x: "Thank you!")),
    (lambda x: x.sentiment == "negative", RunnableLambda(lambda x: "Sorry to hear that.")),
    RunnableLambda(lambda x: "Could not determine the sentiment."),
)

for text, canned in (("This is a beautiful phone", '{"sentiment": "positive"}'),
                     ("This is a terrible phone",  '{"sentiment": "negative"}')):
    c = (classify | FakeListChatModel(responses=[canned]) | parser2) | branch
    print(f"{text!r:<30} -> {c.invoke({'feedback': text})}")

In [ ]:
# Now break it on purpose. A model that answers "Positive" is CORRECT English
# and misses the comparison entirely.
class Loose(BaseModel):
    sentiment: str

for value in ("positive", "Positive", "This is positive feedback"):
    print(f"{value!r:<30} -> {branch.invoke(Loose(sentiment=value))}")

print()
print("Only the exact lowercase string hits a branch. The other two fall")
print("through to the default, the app does something harmless and useless,")
print("and there is no error anywhere to find.")
print()
print("THAT is why the classifier uses Literal['positive','negative'] in a")
print("Pydantic schema - it makes the value exact.")

## Part E — Looking at a chain

In [ ]:
g = chain.get_graph()
print(len(g.nodes), "nodes,", len(g.edges), "edges")
for node in g.nodes.values():
    print("   ", node.name)

assert len(g.nodes) == 5      # 3 components + input + output

In [ ]:
# Most tutorials show this. It needs a package LangChain does not install.
try:
    chain.get_graph().print_ascii()
except ImportError as e:
    print("ImportError:", e)
    print()
    print("Either `pip install grandalf`, or read the node list above,")
    print("which needs nothing extra.")

## What to take away

- `prompt | model | parser` is a **`RunnableSequence`** you can inspect.
- **Sequential** chains just get longer; the parser between two model calls is
  what keeps the types lining up.
- **Parallel** measured **1.99×** on two calls. The ceiling is the branch count,
  and it saves **latency, not cost**.
- **Conditional** runs exactly one branch — and fails *silently* if the
  classifier is not pinned to an exact value.
- A 3-step chain's graph has **5 nodes**; `print_ascii()` needs `grandalf`.

## Exercises

1. Add a third branch to the parallel chain. Does the speedup approach 3×?
   Where does the remaining gap go?
2. Make one branch much slower than the other. What does the total become, and
   what does that tell you about which branch to optimise?
3. Part D's `Loose` model let bad values through. Write a `RunnableLambda` that
   normalises the string (lowercase, strip) before the branch, and decide
   whether you would ship that or the schema.
4. Build a chain with a `RunnableBranch` *inside* a `RunnableParallel`. Does it
   work? What does the output look like?
5. Count the nodes in the graph of the 6-step sequential chain from Part B.
   Predict the number before you run it.